# 1. What you'll learn

        - connect k to boundary smoothness and bias
- demonstrate why distance models require scaling
- measure the curse of dimensionality through distance contrast

        **The one question this notebook answers:** When does remembering nearby examples work, and why do units and extra dimensions quietly destroy the idea of nearby?

# 2. Setup

k-NN has almost no training phase, making preprocessing and geometry unusually visible. We use pipelines so the scaler is learned only on training rows.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Two informative features form three local clusters, then one feature is expressed in units 100 times larger. The class is the local region; later we add pure-noise dimensions.

**Small example:** If a query's five neighbour labels are [A,A,B,A,B], k=1 predicts the closest label while k=5 votes A by 3–2. Larger k smooths individual accidents but can wash out small regions.

In [ ]:
X,y=make_classification(n_samples=900,n_features=2,n_redundant=0,n_informative=2,n_clusters_per_class=2,class_sep=1.15,flip_y=.04,random_state=SEED)
X[:,0]*=100;frame=pd.DataFrame(X,columns=["sensor_millivolts","ratio"]).assign(class_id=y)
print("Shape:",frame.shape,"balance:",frame.class_id.value_counts(normalize=True).round(3).to_dict());print(frame.describe().loc[["mean","std"]].round(2))

# 4. Explore

We compare the raw geometry, neighbour distances around one query, and the distribution of feature scales that will determine an unscaled distance.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
scaled=StandardScaler().fit_transform(X);q=scaled[0];dist=np.linalg.norm(scaled-q,axis=1);near=np.argsort(dist)[1:6]
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].scatter(X[:,0],X[:,1],c=y,cmap="coolwarm",s=12);axes[0].set_title("Raw units stretch x")
axes[1].scatter(scaled[:,0],scaled[:,1],c=y,cmap="coolwarm",s=10,alpha=.3);axes[1].scatter(scaled[near,0],scaled[near,1],s=80,facecolor="none",edgecolor="black");axes[1].scatter(*q,marker="*",s=130,color="gold",edgecolor="black");axes[1].set_title("Five neighbours after scaling")
axes[2].bar(frame.columns[:2],frame.iloc[:,:2].std(),color=["#176b66","#d38b45"]);axes[2].set(title="Raw standard deviations",ylabel="units");plt.tight_layout();plt.show()

**Takeaway:** Raw Euclidean distance is almost entirely the millivolt coordinate.

**Takeaway:** After scaling, a neighbourhood can use both coordinates.

**Takeaway:** A 100× standard-deviation difference is a unit choice, not 100× more predictive information.

# 5. Prepare

We split first and fit StandardScaler only inside the training pipeline. Looking at all rows to estimate means and standard deviations would leak the held-out distribution.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.25,stratify=y,random_state=SEED)
ks=[1,3,7,15,31,61];models={k:make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=k,weights="distance")) for k in ks}
print("Train/test:",X_train.shape,X_test.shape)

# 6. Train

        Training stores the reference cases. Prediction computes distances, selects k nearby cases, and votes; k is therefore the main bias-versus-variance control.

        Conceptually, fitting follows these steps:

        1. Learn training-set center and scale.
2. Store scaled training vectors and labels.
3. For a query, compute its distance to every stored vector.
4. Select k smallest distances and combine their labels, optionally weighting nearer votes more.

In [ ]:
for model in models.values(): model.fit(X_train,y_train)
scores={k:model.score(X_test,y_test) for k,model in models.items()};best_k=max(scores,key=scores.get);model=models[best_k]
print("Test accuracy by k:",{k:round(v,3) for k,v in scores.items()},"chosen k:",best_k)

# 7. Evaluate

We show the chosen boundary, accuracy across k, and empirical distance contrast as independent noise dimensions are added. Majority prediction is the baseline.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
baseline=max(np.mean(y_test),1-np.mean(y_test));dims=[2,5,10,25,50,100];contrasts=[]
rng=np.random.default_rng(SEED)
for d in dims:
    Z=rng.normal(size=(500,d));ds=np.linalg.norm(Z[1:]-Z[0],axis=1);contrasts.append((ds.max()-ds.min())/ds.min())
sx=np.linspace(X[:,0].min(),X[:,0].max(),160);sy=np.linspace(X[:,1].min(),X[:,1].max(),160);xx,yy=np.meshgrid(sx,sy);zz=model.predict(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].contourf(xx,yy,zz,alpha=.35,cmap="coolwarm");axes[0].scatter(X_test[:,0],X_test[:,1],c=y_test,cmap="coolwarm",s=12);axes[0].set_title(f"k={best_k}, accuracy {scores[best_k]:.3f}")
axes[1].plot(ks,list(scores.values()),marker="o");axes[1].axhline(baseline,ls="--",color="black",label=f"baseline {baseline:.2f}");axes[1].set(title="Neighbour count controls smoothing",xlabel="k",ylabel="accuracy");axes[1].legend()
axes[2].plot(dims,contrasts,marker="o",color="#d38b45");axes[2].set(title="Distances become alike",xlabel="dimensions",ylabel="(farthest-nearest)/nearest");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** A moderate k smooths noisy local labels, but the shrinking distance contrast shows why neighbourhoods stop being distinctive as irrelevant dimensions accumulate.

# 8. Break it

We remove scaling while retaining the 100× unit mismatch. The model now mostly sorts points by millivolts, ignoring useful variation in the ratio.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
raw=KNeighborsClassifier(n_neighbors=best_k,weights="distance").fit(X_train,y_train);scaled_model=models[best_k]
print(f"Without scaling={raw.score(X_test,y_test):.3f}; with scaling={scaled_model.score(X_test,y_test):.3f}; damage={scaled_model.score(X_test,y_test)-raw.score(X_test,y_test):+.3f}")

**Use this when…** similar cases should have similar outcomes, the dataset is moderate, dimensions are limited, and a useful metric can be defined.

        **Don't use this when…** features are high-dimensional or semantically incomparable, prediction latency matters, or extrapolation beyond stored examples is required.

        ## Try this

        1. Switch weights from distance to uniform; isolated close examples should matter less.
2. Add 50 Gaussian noise columns and remeasure accuracy.
3. Replace Euclidean with Manhattan distance and inspect which boundary corners change.